In [ ]:
from wsi.nets.vision_transformer import *
from wsi.nets.dino import *
from wsi.utils import get_logger
import torch
from tqdm import tqdm
from wsi.data import DINO_DataSet
import sys


In [ ]:
wsi_list = []
for path,dirs,files in os.walk("../../CAMELYON16/training/"):
    if len(dirs)==0:
        for f in files:
            wsi_list.append(os.path.join(path,f))
len(wsi_list)


In [ ]:
out_dim = 65536
lr = 0.0005
min_lr = 1e-6
epochs = 10
start_epoch = 0
niter_per_wsi = 1000
niter_per_ep = niter_per_wsi*len(wsi_list)
warmup_epochs = 1
weight_decay = 0.04
weight_decay_end = 0.4
momentum_teacher = 0.996
momentum_teacher_end = 1
batch_size = 64
num_workers = 0
clip_grad = 3.0
freeze_last_layer = 1
save_path = "./model/DINO/1_2_3三尺度_1000/"

# Backbone
student = vit_small(drop_path_rate=0.1)
teacher = vit_small()
student = MultiCropWrapper(student,DINOHead(student.embed_dim,out_dim))
teacher = MultiCropWrapper(teacher,DINOHead(teacher.embed_dim, out_dim))

state_dict = torch.load("./model/base/20_checkpoint.pth")
student_state_dict = state_dict["student"]
student_state_dict = {k.replace("module.", ""): v for k, v in student_state_dict.items()}
teacher_state_dict = state_dict["teacher"]
teacher_state_dict = {k.replace("module.", ""): v for k, v in teacher_state_dict.items()}
student.load_state_dict(student_state_dict)
teacher.load_state_dict(teacher_state_dict)

student, teacher = student.cuda(), teacher.cuda()

# Loss
dino_loss = DINOLoss(out_dim,10).cuda()
epoch_loss_best = np.inf

# Optimizer
params_groups = get_params_groups(student)
optimizer = torch.optim.AdamW(params_groups)

# 混合精度训练
fp16_scaler = torch.cuda.amp.GradScaler()

# Scheduler
lr_schedule = cosine_scheduler(lr, min_lr, epochs, niter_per_ep,warmup_epochs)
wd_schedule = cosine_scheduler(weight_decay, weight_decay_end,epochs, niter_per_ep)
momentum_schedule = cosine_scheduler(momentum_teacher, momentum_teacher_end, epochs, niter_per_ep)



# Training
logger = get_logger(save_path)
it_count = 0
for epoch in range(start_epoch, epochs):
    loss_all = []
    logger.info(f"epoche: {epoch} start...")


    for wsi in wsi_list:
        loss_list = []
        dataset = DINO_DataSet(wsi,[1,2,3],max_data_num=niter_per_wsi)
        data_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, num_workers=num_workers)
        for images in tqdm(data_loader):
            # 更新学习率
            for i, param_group in enumerate(optimizer.param_groups):
                param_group["lr"] = lr_schedule[it_count]
                if i == 0:  # only the first group is regularized
                        param_group["weight_decay"] = wd_schedule[it_count]

            # 预测计算损失
            images = [im.cuda(non_blocking=True) for im in images]
            with torch.cuda.amp.autocast(fp16_scaler is not None):
                teacher_output = teacher(images[:2])  # only the 2 global views pass through the teacher
                student_output = student(images)
                loss = dino_loss(student_output, teacher_output, epoch)        
            if not math.isfinite(loss.item()):
                    print("Loss is {}, stopping training".format(loss.item()), force=True)
                    sys.exit(1)
            loss_list.append(loss.item())

            # 更新学生网络
            optimizer.zero_grad()
            fp16_scaler.scale(loss).backward()
            fp16_scaler.unscale_(optimizer)
            clip_gradients(student, clip_grad)
            cancel_gradients_last_layer(epoch, student, freeze_last_layer)
            fp16_scaler.step(optimizer)
            fp16_scaler.update()

            # EMA 更新教师网络
            with torch.no_grad():
                m = momentum_schedule[it_count]  # momentum parameter
                for param_q, param_k in zip(student.parameters(), teacher.parameters()):
                    param_k.data.mul_(m).add_((1 - m) * param_q.detach().data)
            
            it_count += 1
        loss_all.append(np.mean(loss_list))
        logger.info(f"loss_epoch:{np.mean(loss_all)}\tloss_wsi:{loss_all[-1]}")
        
        save_dict = {
            'student': student.state_dict(),
            'teacher': teacher.state_dict(),
            'optimizer': optimizer.state_dict(),
            'epoch': epoch + 1,
            'dino_loss': dino_loss.state_dict(),
            "fp16_scaler" : fp16_scaler
        }
        torch.save(save_dict,os.path.join(save_path,"checkpoint.pth"))

    loss_all = np.mean(loss_all)
    if loss_all<epoch_loss_best:
        epoch_loss_best=loss_all
        torch.save(save_dict,os.path.join(save_path,"checkpoint_epo_best.pth"))


